# Instructor · prepare the bundled sample scene (run once)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week0_Prepare_Sample_Data.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

Creates `data/taipei_s2_sample.tif` and `data/taipei_s2_sample_scl.tif` — one **real** Sentinel-2 L2A window over Taipei (6 bands, float32 reflectance, EPSG:32651, 10 m) plus its Sen2Cor class map — that every course notebook falls back to when Earth Search is unreachable.

Run in Colab (has AWS access), download the two files, put them in the repo's `data/` folder, commit, push. About two minutes.

In [ ]:
# --- 0 · environment (Colab: ~1 minute the first time) ---
import importlib, subprocess, sys
for pkg, mod in [("rasterio", "rasterio"), ("pystac-client", "pystac_client"), ("scikit-image", "skimage"), ("opencv-python-headless", "cv2")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, json, time, urllib.request, numpy as np, matplotlib.pyplot as plt, rasterio, cv2, skimage
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.transform import from_origin
print("python", sys.version.split()[0], "| numpy", np.__version__, "| rasterio", rasterio.__version__,
      "| GDAL", rasterio.__gdal_version__, "| OpenCV", cv2.__version__, "| scikit-image", skimage.__version__)

In [ ]:
os.environ['DL_SPACE_FORCE_TIER'] = 'live'   # this notebook must use the live catalogue

# --- 1 · get a real scene: four tiers, first one that works wins ---
REPO_RAW   = "https://raw.githubusercontent.com/trongan93/dl-space-2026/main/data/"
BBOX       = [121.35, 25.00, 121.65, 25.20]   # Tamsui river mouth -> Taipei basin (WGS84)
DATE_RANGE = "2026-01-01/2026-09-20"
MAX_CLOUD  = 20
GSD        = 10
MAX_PX     = 800                          # window cap (rows/cols) so Colab RAM is safe
BANDS      = {"blue": "B02", "green": "B03", "red": "B04", "nir": "B08", "swir16": "B11", "swir22": "B12"}
BAND_ORDER = list(BANDS); I = {b: i for i, b in enumerate(BAND_ORDER)}
SCL_VALID  = [4, 5, 6, 7, 11]
SCL_NAMES  = {0: "nodata", 1: "saturated", 2: "dark", 3: "cloud shadow", 4: "vegetation", 5: "not vegetated",
              6: "water", 7: "unclassified", 8: "cloud medium", 9: "cloud high", 10: "thin cirrus", 11: "snow/ice"}
FORCE_TIER = os.environ.get("DL_SPACE_FORCE_TIER")   # "live" | "sample" | "synthetic" (testing only)

def tier_upload():
    """Tier 0: the student's own Lab 0 output in the working directory."""
    if FORCE_TIER or not (os.path.exists("lab0_cube.tif") and os.path.exists("lab0_scl.npy")):
        raise FileNotFoundError("no lab0_cube.tif / lab0_scl.npy here")
    with rasterio.open("lab0_cube.tif") as src:
        cube, crs, tr, tags = src.read().astype("float32"), src.crs, src.transform, src.tags()
    return cube, np.load("lab0_scl.npy"), crs, tr, tags.get("scene_id", "lab0_cube.tif (your Lab 0)"), "upload"

def tier_live():
    """Tier 1: search Earth Search (STAC) and read one window from the cloud-optimised GeoTIFFs."""
    if FORCE_TIER not in (None, "live"): raise RuntimeError("tier skipped")
    from pystac_client import Client
    cat = Client.open("https://earth-search.aws.element84.com/v1")
    items = list(cat.search(collections=["sentinel-2-l2a"], bbox=BBOX, datetime=DATE_RANGE,
                            query={"eo:cloud_cover": {"lt": MAX_CLOUD}}, max_items=40).item_collection())
    if not items: raise RuntimeError("no scene matched")
    items.sort(key=lambda i: i.properties["eo:cloud_cover"]); item = items[0]
    with rasterio.open(item.assets["red"].href) as src: crs = src.crs
    b = transform_bounds("EPSG:4326", crs, *BBOX)
    W = min(int(round((b[2] - b[0]) / GSD)), MAX_PX); H = min(int(round((b[3] - b[1]) / GSD)), MAX_PX)
    b = (b[0], b[3] - H * GSD, b[0] + W * GSD, b[3])
    def read(href, resampling):
        with rasterio.open(href) as src:
            win = from_bounds(*b, transform=src.transform)
            return src.read(1, window=win, out_shape=(H, W), resampling=resampling, boundless=True, fill_value=0)
    layers = []
    for k in BAND_ORDER:
        dn = read(item.assets[k].href, Resampling.bilinear)
        rho = (dn.astype("float32") - 1000) / 10000; rho[dn == 0] = np.nan; layers.append(rho)
    scl = read(item.assets["scl"].href, Resampling.nearest).astype("uint8")
    tags = dict(datetime=item.properties["datetime"], cloud=item.properties["eo:cloud_cover"],
                pb=item.properties.get("s2:processing_baseline", "?"), tile=item.properties.get("grid:code", "?"))
    return np.stack(layers), scl, crs, from_origin(b[0], b[3], GSD, GSD), item.id, ("live", tags)

def tier_sample():
    """Tier 2: the course's bundled real Sentinel-2 window (data/ in the repo, or downloaded from GitHub)."""
    if FORCE_TIER not in (None, "sample"): raise RuntimeError("tier skipped")
    paths = {}
    for fn in ["taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"]:
        for cand in [fn, os.path.join("data", fn), os.path.join("..", "data", fn)]:
            if os.path.exists(cand): paths[fn] = cand; break
        else:
            urllib.request.urlretrieve(REPO_RAW + fn, fn); paths[fn] = fn
    with rasterio.open(paths["taipei_s2_sample.tif"]) as src:
        cube, crs, tr, tags = src.read().astype("float32"), src.crs, src.transform, src.tags()
    with rasterio.open(paths["taipei_s2_sample_scl.tif"]) as src: scl = src.read(1).astype("uint8")
    return cube, scl, crs, tr, tags.get("scene_id", "bundled sample"), "sample"

def tier_synthetic():
    """Tier 3: a labelled synthetic scene so every cell still runs. NOT real data."""
    rng = np.random.default_rng(0); H, W = 600, 800
    yy, xx = np.mgrid[0:H, 0:W] / 100.0
    river = np.abs(yy - 3 - 1.2 * np.sin(xx / 1.5)) < 0.25; sea = xx < 1.2 + 0.3 * np.sin(yy)
    lake = ((xx - 6.5) ** 2 + (yy - 4.8) ** 2) < 0.35
    water = river | sea | lake; urban = (xx > 3.5) & (yy > 3.5) & ~water
    def band(wv, vv, uv, n=0.02):
        a = np.where(water, wv, np.where(urban, uv, vv)).astype("float32"); return np.clip(a + rng.normal(0, n, a.shape), 0.001, 0.95)
    cube = np.stack([band(.06,.04,.12), band(.08,.07,.14), band(.05,.05,.16), band(.02,.45,.22), band(.01,.25,.30), band(.005,.15,.28)])
    cloud = ((xx - 6) ** 2 + (yy - 1.5) ** 2) < 0.6; cube[:, cloud] = 0.7
    scl = np.where(water, 6, np.where(urban, 5, 4)).astype("uint8"); scl[cloud] = 9
    cube[:, :15, :] = np.nan; scl[:15, :] = 0
    return cube, scl, rasterio.crs.CRS.from_epsg(32651), from_origin(290000, 2790000, GSD, GSD), "SYNTHETIC-SCENE - not real data", "synthetic"

t0 = time.time(); SOURCE = None
for fn in (tier_upload, tier_live, tier_sample, tier_synthetic):
    try:
        cube, scl, cube_crs, cube_transform, SCENE_ID, SOURCE = fn(); break
    except Exception as e:
        print(f"  {fn.__name__:15s} -> skipped: {repr(e)[:90]}")
if isinstance(SOURCE, tuple): SOURCE, live_tags = SOURCE; print("  live scene:", live_tags)
valid = np.isin(scl, SCL_VALID) & np.isfinite(cube).all(0)
print(f"\nSCENE: {SCENE_ID}\nsource tier: {SOURCE} | cube (C,H,W) = {cube.shape} {cube.dtype} | CRS {cube_crs} | {time.time()-t0:.1f} s")
print(f"reflectance range (valid): {np.nanmin(cube[:, valid]):.3f} - {np.nanmax(cube[:, valid]):.3f} | valid fraction {100*valid.mean():.1f} %")
if SOURCE == "synthetic": print("\n>>> SYNTHETIC scene: fine for following the demo, NOT valid for any submission.")

In [ ]:
assert SOURCE == "live", "Live tier failed - check network / Earth Search status and rerun"
os.makedirs("data", exist_ok=True)
tags = dict(scene_id=SCENE_ID, datetime=live_tags["datetime"], cloud_cover=str(live_tags["cloud"]), processing_baseline=str(live_tags["pb"]),
            tile=str(live_tags["tile"]), bbox_wgs84=str(BBOX), decode="rho=(DN-1000)/10000 applied; values are surface reflectance",
            bands=",".join(f"{k}={BANDS[k]}" for k in BAND_ORDER), source="Copernicus Sentinel data 2026, via Earth Search (Element 84 / AWS)",
            course="Deep Learning in Space Technology Applications, NTUT 115-1")
prof = dict(driver="GTiff", height=cube.shape[1], width=cube.shape[2], crs=cube_crs, transform=cube_transform,
            compress="deflate", predictor=2, tiled=True, blockxsize=256, blockysize=256)
with rasterio.open("data/taipei_s2_sample.tif", "w", dtype="float32", count=6, nodata=np.nan, **prof) as dst:
    dst.write(cube); dst.descriptions = tuple(f"{k} {BANDS[k]}" for k in BAND_ORDER); dst.update_tags(**tags)
    dst.build_overviews([2, 4, 8], Resampling.average)
with rasterio.open("data/taipei_s2_sample_scl.tif", "w", dtype="uint8", count=1, nodata=0, **prof) as dst:
    dst.write(scl, 1); dst.update_tags(**tags, layer="SCL Sen2Cor scene classification")
for fn in ["taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"]:
    print(fn, "%.1f MB" % (os.path.getsize("data/" + fn) / 1e6))
with rasterio.open("data/taipei_s2_sample.tif") as src: print(src.profile["crs"], src.shape, src.tags()["scene_id"], "PB", src.tags()["processing_baseline"])

Quick look, then download.

In [ ]:
def stretch(img, lo=2, hi=98):
    out = np.empty_like(img)
    for i in range(img.shape[-1]):
        a, b = np.nanpercentile(img[..., i], [lo, hi]); out[..., i] = np.clip((img[..., i] - a) / (b - a + 1e-9), 0, 1)
    return np.nan_to_num(out)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(stretch(np.stack([cube[I["red"]], cube[I["green"]], cube[I["blue"]]], -1))); ax[0].set_title(SCENE_ID); ax[0].axis("off")
ax[1].imshow(scl, cmap="tab20", vmin=0, vmax=11); ax[1].set_title("SCL"); ax[1].axis("off"); plt.tight_layout(); plt.show()
try:
    from google.colab import files
    for fn in ["taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"]: files.download("data/" + fn)
except ImportError:
    print("not in Colab - files are in ./data/")

## Then, on your Mac
```bash
cd ~/dl-space-2026            # your clone of https://github.com/trongan93/dl-space-2026
mv ~/Downloads/taipei_s2_sample.tif ~/Downloads/taipei_s2_sample_scl.tif data/
git add data/ && git commit -m "Add bundled Sentinel-2 Taipei sample" && git push
```
The two files should total well under 25 MB (GitHub's per-file limit is 100 MB). Re-run this notebook whenever you want a fresher scene; the notebooks read the file's tags, so nothing else changes.